Tahap 1

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.inspection import permutation_importance
import joblib

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

print("Semua pustaka berhasil dimuat.")

In [ ]:
df_raw = pd.read_csv("Raw-Final-Rumah.csv")

df = df_raw.copy(deep=True)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

In [ ]:
df.duplicated().sum()

In [ ]:
for col in df.columns:
  print(f"Unique values in column '{col} : ")
  print(df[col].unique())
  print("-"*30)

In [ ]:
suspects = [
    "?", "??", "NA", "N/A", "na", "None", "null",
    "Null", "Unknown", "UNK", "-", "--", " ", "Nan",
    "XXX", "unk"
]

for val in suspects:
  print(f"{val}:")
  print((df == val).sum())

In [ ]:
kolom_diperiksa = [
    "Kamar Tidur",
    "Kamar Mandi",
    "Garasi",
    "Luas Tanah",
    "Luas Bangunan"
]

for col in kolom_diperiksa:
    print(f"\n--- {col}: contoh nilai ---")
    display(
        df[col]
        .dropna()
        .astype(str)
        .sample(min(15, df[col].notna().sum()), random_state=42)
        .to_frame()
    )

In [ ]:
def parse_harga(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace("\u00a0", " ")
    match = re.search(r"(\d[\d.,]*)\s*(miliar|milyar|juta|jt)\b", text)
    if not match:
        return np.nan
    angka_text, satuan = match.group(1), match.group(2)
    if "," in angka_text:
        angka_text = angka_text.replace(".", "").replace(",", ".")
    elif angka_text.count(".") > 1:
        angka_text = angka_text.replace(".", "")
    angka = float(angka_text)
    if satuan in ["miliar", "milyar"]:
        return angka * 1_000_000_000
    return angka * 1_000_000


def parse_angka(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Hanya menerima isi berupa angka, bukan teks campuran.
    if re.fullmatch(r"\d+(?:[.,]\d+)?", text):
        return float(text.replace(",", "."))

    return np.nan


def parse_luas(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()
    match = re.search(r"(\d+(?:[.,]\d+)?)\s*m\u00b2", text)

    if not match:
        return np.nan

    return float(match.group(1).replace(",", "."))

In [ ]:
df_eda = df.copy()

df_eda["Harga_Rp"] = df_eda["Harga"].apply(parse_harga)

for col in ["Kamar Tidur", "Kamar Mandi", "Garasi"]:
    df_eda[col + "_num"] = df_eda[col].apply(parse_angka)

for col in ["Luas Tanah", "Luas Bangunan"]:
    df_eda[col + "_num"] = df_eda[col].apply(parse_luas)

kolom_numerik_bersih = [
    "Harga_Rp",
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]

display(df_eda[kolom_numerik_bersih].describe().T)

In [ ]:
harga_valid = df_eda["Harga_Rp"].dropna()
harga_positif = harga_valid[harga_valid > 0]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot distribusi asli
sns.histplot(harga_positif, bins=40, kde=True, ax=axes[0], color="#2563eb")
axes[0].set_title("Distribusi Harga Asli (Sangat Skewed)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Harga (Rp)")

# Plot distribusi log10
sns.histplot(np.log10(harga_positif), bins=40, kde=True, ax=axes[1], color="#059669")
axes[1].set_title("Distribusi Log10 Harga (Mendekati Normal/Gaussian)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Log10 Harga (Rp)")

plt.tight_layout()
plt.show()

In [ ]:
fitur_plot = [
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]

df_eda[fitur_plot].hist(figsize=(14, 9), bins=30)
plt.suptitle("Distribusi Fitur Rumah", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
urutan = df_eda["Kecamatan"].value_counts().index

sns.countplot(
    data=df_eda,
    y="Kecamatan",
    order=urutan
)

plt.title("Jumlah Listing per Kecamatan")
plt.xlabel("Jumlah Listing")
plt.ylabel("Kecamatan")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot harga; gunakan harga positif agar skala log valid
plt.figure(figsize=(10, 4))
sns.boxplot(x=np.log10(harga_positif))
plt.title("Pemeriksaan Outlier pada Log10 Harga Rumah")
plt.xlabel("Log10 Harga (Rp)")
plt.tight_layout()
plt.show()

# Boxplot per fitur dengan satuan terpisah
for col in fitur_plot:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df_eda[col])
    plt.title(f"Pemeriksaan Outlier: {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
kolom_korelasi = [
    "Harga_Rp",
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]

pearson_matrix = df_eda[kolom_korelasi].corr(method="pearson")

plt.figure(figsize=(9, 7))
sns.heatmap(
    pearson_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)
plt.title("Korelasi Pearson Fitur Numerik dan Harga")
plt.tight_layout()
plt.show()

In [ ]:
spearman_matrix = df_eda[kolom_korelasi].corr(method="spearman")

plt.figure(figsize=(9, 7))
sns.heatmap(
    spearman_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)
plt.title("Korelasi Spearman Fitur Numerik dan Harga")
plt.tight_layout()
plt.show()

In [ ]:
ringkasan_kecamatan = (
    df_eda.dropna(subset=["Harga_Rp"])
    .groupby("Kecamatan")["Harga_Rp"]
    .agg(Jumlah_Listing="count", Median="median", Rata_Rata="mean", Min="min", Max="max")
    .sort_values("Median", ascending=False)
)
display(ringkasan_kecamatan)

In [ ]:
plt.figure(figsize=(10, 6))
ringkasan_kecamatan["Median"].sort_values().plot(kind="barh", color="#3b82f6")
plt.title("Median Harga Properti Berdasarkan Kecamatan (Rp)", fontsize=14, fontweight="bold")
plt.xlabel("Median Harga (Rp)")
plt.ylabel("Kecamatan")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import kruskal

kelompok_harga = [
    grup["Harga_Rp"].dropna().to_numpy()
    for _, grup in df_eda.groupby("Kecamatan")
]

kelompok_harga = [g for g in kelompok_harga if len(g) > 0]

if len(kelompok_harga) >= 2:
    statistik, p_value = kruskal(*kelompok_harga)
    print("Statistik Kruskal-Wallis:", statistik)
    print("P-value:", p_value)

Tahap 2

In [ ]:
def parse_area_label(value):
    if pd.isna(value):
        return None, np.nan
    text = str(value).strip().lower()
    match = re.search(r"\b(lt|lb)\b\s*:?\s*(\d+(?:[.,]\d+)?)\s*m(?:\u00b2|2)?\b", text)
    if not match:
        return None, np.nan
    return match.group(1).upper(), float(match.group(2).replace(",", "."))

In [ ]:
for value in [
    "LT : 438 m\u00b2",
    "LB : 110 m\u00b2",
    "LB:41 m\u00b2",
    "LT:1000 m\u00b2"
]:
    print(value, "=>", parse_area_label(value))

In [ ]:
def repair_area_row(row):
    sumber = ["Kamar Tidur", "Kamar Mandi", "Garasi", "Luas Tanah", "Luas Bangunan"]
    kandidat_lt, kandidat_lb = [], []
    for col in sumber:
        label, nilai = parse_area_label(row[col])
        if label == "LT":
            kandidat_lt.append(nilai)
        elif label == "LB":
            kandidat_lb.append(nilai)
    
    def pilih_nilai(kandidat):
        nilai_unik = sorted(set(kandidat))
        if len(nilai_unik) == 1:
            return nilai_unik[0], False
        if len(nilai_unik) == 0:
            return np.nan, False
        return np.nan, True
        
    lt, konflik_lt = pilih_nilai(kandidat_lt)
    lb, konflik_lb = pilih_nilai(kandidat_lb)
    return pd.Series({
        "LT_recovered": lt,
        "LB_recovered": lb,
        "Konflik_LT": konflik_lt,
        "Konflik_LB": konflik_lb
    })


print("Menjalankan Regex Data Recovery pada baris data...")
hasil_repair = df_raw.apply(repair_area_row, axis=1)

df_clean = df_raw.copy()

df_clean["LT_recovered"] = hasil_repair["LT_recovered"]
df_clean["LB_recovered"] = hasil_repair["LB_recovered"]
df_clean["Konflik_LT"] = hasil_repair["Konflik_LT"]
df_clean["Konflik_LB"] = hasil_repair["Konflik_LB"]

display(
    df_clean[
        [
            "Kamar Tidur",
            "Kamar Mandi",
            "Garasi",
            "Luas Tanah",
            "Luas Bangunan",
            "LT_recovered",
            "LB_recovered",
            "Konflik_LT",
            "Konflik_LB"
        ]
    ].head(15)
)

In [ ]:
print(
    "Luas tanah berhasil dipulihkan:",
    df_clean["LT_recovered"].notna().sum()
)

print(
    "Luas bangunan berhasil dipulihkan:",
    df_clean["LB_recovered"].notna().sum()
)

print(
    "Baris dengan konflik LT:",
    df_clean["Konflik_LT"].sum()
)

print(
    "Baris dengan konflik LB:",
    df_clean["Konflik_LB"].sum()
)

In [ ]:
for col in ["Kamar Tidur", "Kamar Mandi", "Garasi"]:
    df_clean[col + "_num"] = df_clean[col].apply(parse_angka)

df_clean["Harga_Rp"] = df_clean["Harga"].apply(parse_harga)

df_clean["Luas Tanah_num"] = df_clean["LT_recovered"]
df_clean["Luas Bangunan_num"] = df_clean["LB_recovered"]

In [ ]:
kolom_num = [
    "Harga_Rp",
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]

display(df_clean[kolom_num].describe().T)

In [ ]:
audit = pd.DataFrame({
    "Missing Sebelum": df_raw.isna().sum(),
    "Missing Sesudah Parsing": df_clean[
        [
            "Harga_Rp",
            "Kamar Tidur_num",
            "Kamar Mandi_num",
            "Garasi_num",
            "Luas Tanah_num",
            "Luas Bangunan_num"
        ]
    ].isna().sum().reindex([
        "Harga_Rp",
        "Kamar Tidur_num",
        "Kamar Mandi_num",
        "Garasi_num",
        "Luas Tanah_num",
        "Luas Bangunan_num"
    ])
})

display(audit)

In [ ]:
print("Duplikat persis:", df_raw.duplicated().sum())

In [ ]:
df_unique = df_raw.drop_duplicates().copy()

print("Baris sebelum:", len(df_raw))
print("Baris sesudah:", len(df_unique))
print("Baris dihapus:", len(df_raw) - len(df_unique))

In [ ]:
for col in [
    "Harga_Rp",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]:
    values = df_clean[col].dropna()

    plt.figure(figsize=(9, 3))
    plt.boxplot(values, vert=False)
    plt.title(f"Boxplot {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
def iqr_outlier_summary(series):
    s = series.dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (s < lower) | (s > upper)

    return {
        "Q1": q1,
        "Q3": q3,
        "Batas Bawah": lower,
        "Batas Atas": upper,
        "Jumlah Kandidat Outlier": int(mask.sum())
    }


for col in [
    "Harga_Rp",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]:
    print(col, iqr_outlier_summary(df_clean[col]))

Tahap 3

In [ ]:
# Filter Domain Properti Wajar
mask_harga = (df_clean["Harga_Rp"] >= 150_000_000) & (df_clean["Harga_Rp"] <= 20_000_000_000)
mask_lt = df_clean["Luas Tanah_num"].isna() | ((df_clean["Luas Tanah_num"] >= 25) & (df_clean["Luas Tanah_num"] <= 1200))
mask_lb = df_clean["Luas Bangunan_num"].isna() | ((df_clean["Luas Bangunan_num"] >= 20) & (df_clean["Luas Bangunan_num"] <= 800))
mask_kt = df_clean["Kamar Tidur_num"].isna() | ((df_clean["Kamar Tidur_num"] >= 1) & (df_clean["Kamar Tidur_num"] <= 12))
mask_km = df_clean["Kamar Mandi_num"].isna() | ((df_clean["Kamar Mandi_num"] >= 1) & (df_clean["Kamar Mandi_num"] <= 10))

df_filtered = df_clean[mask_harga & mask_lt & mask_lb & mask_kt & mask_km].drop_duplicates().copy()

print(f"Data awal sebelum filtering : {df_raw.shape[0]} baris")
print(f"Data bersih setelah filtering: {df_filtered.shape[0]} baris")
print(f"Total baris outlier/duplikat dibuang: {df_raw.shape[0] - df_filtered.shape[0]} baris")

kolom_stat = ["Harga_Rp", "Kamar Tidur_num", "Kamar Mandi_num", "Garasi_num", "Luas Tanah_num", "Luas Bangunan_num"]
display(df_filtered[kolom_stat].describe().T)

Tahap 4

In [ ]:
fitur = [
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num",
    "Kecamatan"
]

X = df_filtered[fitur].copy()
y = df_filtered["Harga_Rp"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f"Dimensi Data Training (X_train): {X_train.shape}")
print(f"Dimensi Data Testing  (X_test) : {X_test.shape}")

In [ ]:
kolom_numerik = ["Kamar Tidur_num", "Kamar Mandi_num", "Garasi_num", "Luas Tanah_num", "Luas Bangunan_num"]
kolom_kategori = ["Kecamatan"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            kolom_numerik
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            kolom_kategori
        )
    ]
)

print("Preprocessor Pipeline berhasil dikonfigurasi:")
display(preprocessor)

Tahap 5

In [ ]:
def evaluasi_model(y_true, y_pred):
    mask = np.asarray(y_true) > 0
    mape = mean_absolute_percentage_error(np.asarray(y_true)[mask], np.asarray(y_pred)[mask]) * 100
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MAE (Rp)": mae, "MAPE (%)": mape, "R2": r2}

candidate_models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=7),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

hasil_benchmark = []
fitted_pipelines = {}

print("Memulai proses pelatihan dan benchmarking model...")
for nama, model in candidate_models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", TransformedTargetRegressor(
            regressor=model,
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    metrik = evaluasi_model(y_test, y_pred)
    metrik["Model"] = nama
    hasil_benchmark.append(metrik)
    fitted_pipelines[nama] = pipe
    print(f"Selesai: {nama:20s} | MAE: Rp {metrik['MAE (Rp)']:12,.0f} | MAPE: {metrik['MAPE (%)']:5.2f}% | R2: {metrik['R2']:.4f}")

df_benchmark = pd.DataFrame(hasil_benchmark)[["Model", "MAE (Rp)", "MAPE (%)", "R2"]].sort_values("R2", ascending=False)
display(df_benchmark)

Tahap 6

In [ ]:
fitur_korelasi = [
    "Harga_Rp",
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num"
]

corr = df_filtered[fitur_korelasi].corr(method="spearman")

plt.figure(figsize=(9, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)
plt.title("Korelasi Spearman antarvariabel")
plt.tight_layout()
plt.show()

display(corr)

Tahap 7

In [ ]:
param_dist = {
    "regressor__regressor__n_estimators": [100, 150, 200],
    "regressor__regressor__max_depth": [15, 20, 25, 30],
    "regressor__regressor__min_samples_split": [2, 5],
    "regressor__regressor__min_samples_leaf": [1, 2]
}

base_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", TransformedTargetRegressor(
        regressor=RandomForestRegressor(random_state=42, n_jobs=-1),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_rf_pipeline,
    param_distributions=param_dist,
    n_iter=8,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

search.fit(X_train, y_train)

print("\nParameter Terbaik:")
print(search.best_params_)
print(f"Best CV MAE (Rp): {-search.best_score_:,.0f}")

In [ ]:
model_final = search.best_estimator_

perm = permutation_importance(
    model_final,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

df_importance = pd.DataFrame({
    "Fitur": X_test.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(df_importance)

plt.figure(figsize=(9, 5))
plt.barh(df_importance["Fitur"][::-1], df_importance["Importance"][::-1], color="#2563eb")
plt.xlabel("Kenaikan Nilai Neg-MAE (Dampak terhadap Penurunan Error)")
plt.title("Permutation Feature Importance (Data Testing)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

Tahap 8

In [ ]:
pred_final = model_final.predict(X_test)
hasil_final = evaluasi_model(y_test, pred_final)

print("=== HASIL EVALUASI MODEL AKHIR (TEST SET) ===")
print(f"MAE  : Rp {hasil_final['MAE (Rp)']:,.0f}")
print(f"MAPE : {hasil_final['MAPE (%)']:.2f}%")
print(f"R\u00b2   : {hasil_final['R2']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Scatter Plot Aktual vs Prediksi
axes[0].scatter(y_test / 1e9, pred_final / 1e9, alpha=0.3, color="#2563eb")
batas_min = min(y_test.min(), pred_final.min()) / 1e9
batas_max = max(y_test.max(), pred_final.max()) / 1e9
axes[0].plot([batas_min, batas_max], [batas_min, batas_max], "r--", linewidth=2)
axes[0].set_xlabel("Harga Aktual (Miliar Rp)", fontsize=11)
axes[0].set_ylabel("Harga Prediksi (Miliar Rp)", fontsize=11)
axes[0].set_title("Harga Aktual vs Prediksi Model (Garis Ideal Merah)", fontsize=13, fontweight="bold")

# 2. Residual Plot
residuals = (y_test - pred_final) / 1e9
axes[1].scatter(pred_final / 1e9, residuals, alpha=0.3, color="#059669")
axes[1].axhline(0, color="r", linestyle="--", linewidth=2)
axes[1].set_xlabel("Harga Prediksi (Miliar Rp)", fontsize=11)
axes[1].set_ylabel("Residual (Aktual - Prediksi) (Miliar Rp)", fontsize=11)
axes[1].set_title("Plot Residual vs Harga Prediksi", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

Tahap 9

In [ ]:
# Menyimpan model ke disk
model_path = "model_harga_rumah.joblib"
joblib.dump(model_final, model_path)
print(f"Model berhasil disimpan ke berkas: {model_path}")

# Verifikasi simulasi pemuatan model seperti di Streamlit
model_loaded = joblib.load(model_path)

sample_input = pd.DataFrame([{
    "Kamar Tidur_num": 3,
    "Kamar Mandi_num": 2,
    "Garasi_num": 1,
    "Luas Tanah_num": 100.0,
    "Luas Bangunan_num": 80.0,
    "Kecamatan": "Cinere"
}])

estimasi_sample = float(model_loaded.predict(sample_input)[0])
print(f"Contoh Prediksi Properti Cinere (LT 100m\u00b2, LB 80m\u00b2, 3KT, 2KM): Rp {estimasi_sample:,.0f}")